In [6]:
# ==========================================
# 1. DEPENDENCY INSTALLATION (Google Colab)
# ==========================================
!pip install -q qiskit==1.0.2 qiskit-aer==0.14.1 qiskit-optimization==0.6.1 qiskit-algorithms==0.3.0 networkx pandas matplotlib pylatexenc qiskit-ibm-runtime

# ==========================================
# 2. IMPORTS & FOLDER STRUCTURE SETUP
# ==========================================
import os
import time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import warnings

from qiskit_aer import AerSimulator
from qiskit_aer.primitives import Sampler as AerSampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2, FakeNairobiV2
from qiskit_optimization.applications import Maxcut
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA

warnings.filterwarnings('ignore')

dirs = {'tables': 'results/tables', 'figures': 'results/figures'}
for path in dirs.values():
    os.makedirs(path, exist_ok=True)

# ==========================================
# 3. PROBLEM DEFINITION (MAX-CUT GRAPH)
# ==========================================
G = nx.Graph()
G.add_edges_from([(0, 1), (1, 2), (2, 3), (3, 4), (4, 0), (0, 2)])

maxcut = Maxcut(G)
qp = maxcut.to_quadratic_program()
observable, offset = qp.to_ising()

# ==========================================
# 4. PROCESSOR CONFIGURATION
# ==========================================
processors = {
    'Processor_A': FakeManilaV2(),
    'Processor_B': FakeNairobiV2()
}

# ==========================================
# 5. BENCHMARKING FUNCTION (QAOA EXECUTION)
# ==========================================
def benchmark_qaoa(processor_name, backend, observable):
    print(f"Benchmarking {processor_name}...")

    simulator = AerSimulator.from_backend(backend)

    sampler = AerSampler(
        backend_options={
            "noise_model": simulator.noise_model,
            "coupling_map": simulator.coupling_map,
            "basis_gates": simulator.operation_names
        },
        run_options={"shots": 1024}
    )

    optimizer = COBYLA(maxiter=50)
    qaoa = QAOA(sampler, optimizer, reps=2)

    start_time = time.time()
    result = qaoa.compute_minimum_eigenvalue(observable)
    execution_time = time.time() - start_time

    best_bitstring = None
    max_prob = 0

    for state_int, prob in result.eigenstate.items():
        bitstring = format(state_int, f'0{observable.num_qubits}b')
        if prob > max_prob:
            max_prob = prob
            best_bitstring = bitstring

    x = [int(bit) for bit in best_bitstring][::-1]
    cut_value = maxcut.objective_value(x)

    return {
        'Processor': processor_name,
        'Execution_Time_sec': round(execution_time, 2),
        'Max_Cut_Value': cut_value
    }

# ==========================================
# 6. RUN EXPERIMENTS & EXPORT DATA
# ==========================================
results_data = []
for name, backend in processors.items():
    results_data.append(benchmark_qaoa(name, backend, observable))

df_results = pd.DataFrame(results_data)
csv_path = f"{dirs['tables']}/PS_O1_benchmark_results.csv"
df_results.to_csv(csv_path, index=False)

print("\n==========================================")
print("BENCHMARK COMPLETE")
print("==========================================")
print(df_results.to_string(index=False))


ERROR: Could not find a version that satisfies the requirement qiskit-aer==0.14.1 (from versions: 0.1.0, 0.2.0, 0.2.1, 0.2.2, 0.2.3, 0.3.0, 0.3.1, 0.3.2, 0.3.3, 0.3.4, 0.4.0, 0.4.1, 0.5.0, 0.5.1, 0.5.2, 0.6.0, 0.6.1, 0.7.0, 0.7.1, 0.7.2, 0.7.3, 0.7.4, 0.7.5, 0.7.6, 0.8.0, 0.8.1, 0.8.2, 0.9.0, 0.9.1, 0.10.0, 0.10.1, 0.10.2, 0.10.3, 0.10.4, 0.11.0, 0.11.1, 0.11.2, 0.12.0, 0.13.0, 0.13.1, 0.13.2, 0.13.3, 0.14.0.1, 0.14.2, 0.15.0, 0.15.1, 0.16.0, 0.16.1, 0.16.2, 0.16.3, 0.16.4, 0.17.0, 0.17.1, 0.17.2)
ERROR: No matching distribution found for qiskit-aer==0.14.1
Benchmarking Processor_A...


AttributeError: 'AerSimulator' object has no attribute 'noise_model'